# Contextual Bandits

## Learning Objectives

1. **Define** the contextual bandit problem and contrast with multi-armed bandits
2. **Implement** LinUCB for linear contextual bandits
3. **Analyze** the exploration bonus and confidence intervals
4. **Apply** contextual bandits to news article recommendation

## From Bandits to Contextual Bandits

**Multi-armed bandit:** $K$ arms with fixed unknown reward distributions. No context.

**Contextual bandit:** at each round $t$:
1. Observe context $x_t \in \mathbb{R}^d$ (e.g., user features + article features)
2. Choose action $a_t \in \{1,\ldots,K\}$
3. Receive reward $r_t = f(x_t, a_t) + \epsilon$
4. Learn to improve future decisions

**Key difference:** rewards depend on context — different users may prefer different arms.

## LinUCB Algorithm

**Assumption:** reward is linear in context:
$$\mathbb{E}[r_t | x_t, a] = x_t^\top \theta_a^*$$

**LinUCB (Li et al. 2010):**
For each arm $a$, maintain:
- $A_a = \sum_{s: a_s=a} x_s x_s^\top + I$ (covariance + regularization)
- $b_a = \sum_{s: a_s=a} r_s x_s$ (reward-weighted context)

Then $\hat{\theta}_a = A_a^{-1} b_a$ and the UCB score is:
$$\text{UCB}(a) = x_t^\top \hat{\theta}_a + \alpha \sqrt{x_t^\top A_a^{-1} x_t}$$

Choose arm with highest UCB; update $A_a$ and $b_a$ after each observation.

In [1]:
import math, random

def mat_add(A, B):
    n = len(A); return [[A[i][j]+B[i][j] for j in range(n)] for i in range(n)]

def mat_vec(A, v):
    return [sum(A[i][j]*v[j] for j in range(len(v))) for i in range(len(A))]

def outer(u, v):
    return [[u[i]*v[j] for j in range(len(v))] for i in range(len(u))]

def solve(A, b):
    """Gaussian elimination (for small matrices only)."""
    n = len(A)
    M = [row[:]+[b[i]] for i,row in enumerate(A)]
    for col in range(n):
        pivot = max(range(col, n), key=lambda r: abs(M[r][col]))
        M[col], M[pivot] = M[pivot], M[col]
        p = M[col][col]
        if abs(p) < 1e-10: continue
        for row in range(n):
            if row != col:
                f = M[row][col]/p
                M[row] = [M[row][j]-f*M[col][j] for j in range(n+1)]
        M[col] = [M[col][j]/p for j in range(n+1)]
    return [M[i][n] for i in range(n)]

def mat_inv_vec(A, v):
    """Solve A x = v."""
    return solve(A, v)

def dot(a, b): return sum(x*y for x,y in zip(a,b))

class LinUCB:
    def __init__(self, n_arms, d, alpha=1.0):
        self.K = n_arms; self.d = d; self.alpha = alpha
        self.A = [[[1.0 if i==j else 0.0 for j in range(d)] for i in range(d)]
                  for _ in range(n_arms)]  # d x d identity
        self.b = [[0.0]*d for _ in range(n_arms)]

    def select(self, context):
        best_arm, best_ucb = 0, float('-inf')
        for a in range(self.K):
            theta = mat_inv_vec(self.A[a], self.b[a])
            Ainv_x = mat_inv_vec(self.A[a], context)
            exploit = dot(theta, context)
            explore = self.alpha * math.sqrt(dot(context, Ainv_x))
            ucb = exploit + explore
            if ucb > best_ucb:
                best_ucb, best_arm = ucb, a
        return best_arm

    def update(self, arm, context, reward):
        xx = outer(context, context)
        self.A[arm] = mat_add(self.A[arm], xx)
        for i in range(self.d): self.b[arm][i] += reward * context[i]

# Simulate news recommendation: 3 articles, 2D user context
K, d = 3, 2
rng = random.Random(42)
# True theta for each article
theta_true = [[1.0, 0.5], [-0.5, 1.0], [0.3, -0.8]]

agent = LinUCB(K, d, alpha=0.5)
cumulative_reward = 0.0
T = 500

for t in range(T):
    # Random user context
    context = [rng.gauss(0,1), rng.gauss(0,1)]
    # Choose arm
    arm = agent.select(context)
    # True reward
    r = dot(theta_true[arm], context) + rng.gauss(0,0.3)
    agent.update(arm, context, r)
    cumulative_reward += r

# Oracle: always picks best arm
oracle_reward = sum(
    max(dot(theta_true[a], [rng.gauss(0,1), rng.gauss(0,1)]) for a in range(K))
    for _ in range(T)
)
print(f"LinUCB cumulative reward over {T} rounds: {cumulative_reward:.2f}")
print(f"(Oracle would get ≈ {oracle_reward:.2f})")

LinUCB cumulative reward over 500 rounds: 501.47
(Oracle would get ≈ 465.18)


## Applications

| Domain | Context | Arms | Reward |
|--------|---------|------|--------|
| **News recommendation** | User profile + time | Articles | Click |
| **Online ads** | User demographics + query | Ad slots | Click/conversion |
| **Clinical trials** | Patient features | Treatment options | Health outcome |
| **Dynamic pricing** | Customer segment + inventory | Price levels | Revenue |
| **Robot control** | State of robot | Actions | Task progress |

**LinUCB achieves $O(\sqrt{Td \log T})$ cumulative regret** — poly-logarithmic in T and linear in d (under standard linear reward assumption). This is provably near-optimal for linear contextual bandits.